# Required Libraries and Functions

In [0]:
import requests, json

In [0]:
dbutils.widgets.text("volume_path", "/Volumes/workspace/taxi_weather/datasets/")

base_path = dbutils.widgets.get("volume_path")

# Hard Code Boroughs and Admin Locations

In [0]:
boroughs = ['EWR', 'Queens', 'Bronx', 'Manhattan', 'Staten Island', 'Brooklyn']
admins = ['New York', 'New Jersey']

# Get Coordinate Data using Open-Meteo's Gecoding API

In [0]:
borough_coords = {}

for borough in boroughs:

    admin = "New Jersey" if borough == "EWR" else "New York"
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={borough},{admin}&count=1"

    response = requests.get(url, timeout=5)

    try:
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        raise RuntimeError(f"Failed to fetch coordinates for {borough}. Error: {str(e)}")

    if "results" in data:
        latitude = data["results"][0]["latitude"]
        longitude = data["results"][0]["longitude"]
        borough_coords[borough] = {
            "latitude": latitude,
            "longitude": longitude
        }
    else:
        raise ValueError(f"No coordinate results found for {borough}")

# Write the data into DBFS Storage

In [0]:
file_path = (f'{base_path}borough_coord.json')

with open(file_path, "w") as f:
    json.dump(borough_coords, f)